# TrafficVision — Entrenamiento YOLOv8n
**Detección de placas vehiculares ecuatorianas**

Pasos:
1. Verificar GPU
2. Instalar dependencias
3. Montar Google Drive
4. Preparar datasets
5. Entrenar modelos
6. Descargar resultados

In [ ]:
# ── CELDA ANTI-DESCONEXIÓN ─────────────────────────────────────
import time, threading

def heartbeat():
    while True:
        time.sleep(45)
        try:
            from google.colab import output
            output.eval_js('document.querySelector("#top-toolbar").click()')
        except:
            pass

thread = threading.Thread(target=heartbeat, daemon=True)
thread.start()
print("✅ Anti-desconexión activo")

✅ Anti-desconexión activo


In [ ]:
#CELDA 1: Verificar GPU
!nvidia-smi
import torch
print(f'\nCUDA disponible: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU"}')

Tue Mar 24 00:47:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
#CELDA 2: Instalar dependencias
!pip install ultralytics -q
from ultralytics import YOLO
print('✅ Ultralytics instalado')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 67.8 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Ultralytics instalado


In [ ]:
# ── CELDA 3: Montar Google Drive ──────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive montado en /content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive montado en /content/drive


In [ ]:
#CELDA 4: Verificar estructura de datasets
import os

# IMPORTANTE: ajustar ruta según donde subiste los datasets
DRIVE_BASE = '/content/drive/MyDrive/TrafficVision/datasets'

datasets = {
    'global':   f'{DRIVE_BASE}/license-plates',
    'ecuador1': f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1',
    'ecuador2': f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-2',
    'ecuador4': f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-4',
}

for name, path in datasets.items():
    exists = os.path.exists(path)
    status = '✅' if exists else '❌'
    if exists:
        train_count = len(os.listdir(f'{path}/train/images')) if os.path.exists(f'{path}/train/images') else 0
        print(f'{status} {name}: {train_count} imágenes de entrenamiento')
    else:
        print(f'{status} {name}: NO ENCONTRADO en {path}')

✅ global: 7057 imágenes de entrenamiento
✅ ecuador1: 54 imágenes de entrenamiento
✅ ecuador2: 90 imágenes de entrenamiento
✅ ecuador4: 375 imágenes de entrenamiento


In [ ]:
#CELDA 5: Crear data.yaml para combined_all
import yaml

DRIVE_BASE = '/content/drive/MyDrive/TrafficVision/datasets'

data = {
    'train': [
        f'{DRIVE_BASE}/license-plates/train/images',
        f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1/train/images',
        f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-2/train/images',
        f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-4/train/images',
    ],
    'val':  f'{DRIVE_BASE}/license-plates/valid/images',
    'test': f'{DRIVE_BASE}/license-plates/test/images',
    'nc':   1,
    'names': ['license plate'],
}

yaml_path = '/content/data_combined_all.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

print('✅ data_combined_all.yaml creado')
print(f'   Train: {len(data["train"])} carpetas')
for p in data['train']:
    count = len(os.listdir(p)) if os.path.exists(p) else 0
    print(f'   - {p.split("/")[-3]}/{p.split("/")[-2]}: {count} imgs')

✅ data_combined_all.yaml creado
   Train: 4 carpetas
   - license-plates/train: 7057 imgs
   - license-plates-ec-1/train: 54 imgs
   - license-plates-ec-2/train: 90 imgs
   - license-plates-ec-4/train: 375 imgs


In [ ]:
#CELDA 6: Entrenar combined_all (global + ecuador)
#Estimado con GPU T4: 30-60 minutos

model = YOLO('yolov8n.pt')

results = model.train(
    data     = '/content/data_combined_all.yaml',
    epochs   = 100,       # más épocas que en CPU
    imgsz    = 640,
    batch    = 32,        # batch más grande con GPU
    name     = 'yolov8n_combined_all',
    project  = '/content/drive/MyDrive/TrafficVision/runs',
    patience = 15,
    save     = True,
    plots    = True,
    device   = 0,         # GPU
    amp      = True,      # Mixed precision — más rápido en GPU
)

print('\n✅ Entrenamiento completado')

Ultralytics 8.4.23 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data_combined_all.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_combined_all, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=1

In [ ]:
#CELDA 7 (OPCIONAL): Entrenar solo Ecuador
import yaml

DRIVE_BASE = '/content/drive/MyDrive/TrafficVision/datasets'

data_ec = {
    'train': [
        f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1/train/images',
        f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-2/train/images',
        f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-4/train/images',
    ],
    'val':  f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1/valid/images',
    'test': f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1/test/images',
    'nc':   1,
    'names': ['license plate'],
}

with open('/content/data_ecuador.yaml', 'w') as f:
    yaml.dump(data_ec, f, default_flow_style=False)

model_ec = YOLO('yolov8n.pt')
model_ec.train(
    data     = '/content/data_ecuador.yaml',
    epochs   = 100,
    imgsz    = 640,
    batch    = 32,
    name     = 'yolov8n_ecuador_combined',
    project  = '/content/drive/MyDrive/TrafficVision/runs',
    patience = 15,
    save     = True,
    plots    = True,
    device   = 0,
    amp      = True,
)

print('\n✅ Entrenamiento Ecuador completado')

Ultralytics 8.4.24 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data_ecuador.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_ecuador_combined, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=15

In [ ]:
# ── CELDA REANUDAR ENTRENAMIENTO ──────────────────────────────
# Usa esto si Colab se desconectó y quieres continuar
import glob, os

RUNS_DIR = "/content/drive/MyDrive/TrafficVision/runs"

# Buscar último checkpoint guardado
last_models = glob.glob(f"{RUNS_DIR}/**/weights/last.pt", recursive=True)

if last_models:
    last_pt = last_models[0]
    print(f"✅ Checkpoint encontrado: {last_pt}")
    size = os.path.getsize(last_pt) / (1024*1024)
    print(f"   Tamaño: {size:.1f} MB")

    # Reanudar desde el último checkpoint
    model = YOLO(last_pt)
    model.train(
        data        = "/content/data_combined_all.yaml",
        epochs      = 100,
        imgsz       = 640,
        batch       = 16,        # ← bajar de 32 a 16 para ahorrar RAM
        name        = "yolov8n_combined_all",
        project     = f"{RUNS_DIR}",
        patience    = 15,
        save        = True,
        save_period = 5,     # guardar cada 5 épocas
        plots       = True,
        device      = 0,
        amp         = True,
        resume      = True,  # ← clave para reanudar
        cache = 'disk',      # ← clave para reanudar
    )
else:
    print("❌ No se encontró checkpoint. Ejecuta la Celda 6 desde el inicio.")

✅ Checkpoint encontrado: /content/drive/MyDrive/TrafficVision/runs/yolov8n_combined_all/weights/last.pt
   Tamaño: 17.5 MB
Ultralytics 8.4.26 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data_combined_all.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/TrafficVision/runs/yolov8n_comb

In [ ]:
# ── CELDA 8: Ver métricas finales ────────────────────────────────
import glob

runs_dir = '/content/drive/MyDrive/TrafficVision/runs'
best_models = glob.glob(f'{runs_dir}/**/weights/best.pt', recursive=True)

print('Modelos entrenados:')
for m in best_models:
    size = os.path.getsize(m) / (1024*1024)
    print(f'  ✅ {m.split("/")[-3]} — {size:.1f} MB')

# Evaluar el mejor modelo
if best_models:
    model_eval = YOLO(best_models[0])
    metrics = model_eval.val(
        data   = '/content/data_combined_all.yaml',
        imgsz  = 640,
        device = 0,
    )
    print(f'\n── Métricas ───────────────────────')
    print(f'  mAP@50:    {metrics.box.map50:.4f}')
    print(f'  mAP@50-95: {metrics.box.map:.4f}')
    print(f'  Precisión: {metrics.box.mp:.4f}')
    print(f'  Recall:    {metrics.box.mr:.4f}')

Modelos entrenados:
  ✅ yolov8n_combined_all — 17.5 MB
  ✅ yolov8n_combined_all3 — 17.5 MB
  ✅ yolov8n_ecuador_combined — 6.0 MB
Ultralytics 8.4.24 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 1.4±1.6 ms, read: 0.0±0.0 MB/s, size: 19.8 KB)
val: Scanning /content/drive/MyDrive/TrafficVision/datasets/license-plates/valid/labels... 2048 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2048/2048 1.1it/s 31:24
val: New cache created: /content/drive/MyDrive/TrafficVision/datasets/license-plates/valid/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 128/128 4.8it/s 26.4s
                   all       2048       2195      0.985      0.942      0.974      0.701
Speed: 0.9ms preprocess, 2.7ms inference, 0.0ms loss, 1.3ms postprocess per image
Results saved to /content/runs/detect/val

── Métric